# 03 · Dim_Cliente
**IDY1102 · EP3 · DataRetail 360**

| Campo | Detalle |
|---|---|
| Responsable | [Nombre del integrante] |
| Tabla destino | `dim_cliente` |
| SCD | **Tipo 2**: comuna de residencia, segmento · Tipo 1: tramo etario, fecha de primer registro, `geografia_key` |
| Clave natural | `rut_anonimizado` (hash del RUT con sal; el RUT nunca se guarda) |
| Depende de | `dim_geografia` (notebook 01) y de la variable `RUT_SALT` en `.env` |
| Fuentes | `clientes_snapshot_1.csv` y `clientes_snapshot_2.csv` |
| Alimenta a | `fact_ventas` (notebook 04) |

> **Cómo usar esta plantilla.** Las celdas marcadas **✏️ TODO** son la parte que debe implementar el integrante
> responsable (las reglas de calidad). Todo lo demás ya está conectado a los módulos compartidos de `src/`
> (carga idempotente, SCD, auditoría, logging). Antes de commitear: *Restart & Run All* debe terminar sin errores.

> ⚠️ **Datos personales:** este notebook maneja RUT, nombre y correo. **No los imprimas** (`head()`, `print`) ni los dejes en las salidas guardadas del notebook: el `.ipynb` se sube a Git. Muestra solo columnas anonimizadas o conteos.

## 0. Setup

In [ ]:
import os, sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db, extract, medallion, quality as q, schema
from src.config import get_settings
from src.logger import get_logger

PROCESO = "dim_cliente"
log = get_logger(PROCESO)
cfg = get_settings()
engine = db.get_engine(cfg)
schema.crear_esquema(engine)                  # idempotente: crea solo lo que falta
schema.sembrar_miembros_desconocidos(engine)  # miembro -1 "No informado" en cada dimensión
pd.set_option("display.width", 160)

## 1. Parámetros

In [ ]:
# Snapshot a procesar: lo define run_pipeline.py (variable de entorno SNAPSHOT) o puedes cambiarlo a mano.
SNAPSHOT = int(os.getenv("SNAPSHOT", "1"))
FECHAS_EFECTIVAS = {1: date(2025, 1, 1), 2: date(2025, 6, 1)}   # desde cuándo rige cada snapshot
FECHA_EFECTIVA = FECHAS_EFECTIVAS[SNAPSHOT]
ARCHIVO = cfg.data_dir / f"clientes_snapshot_{SNAPSHOT}.csv"
print(f"Snapshot {SNAPSHOT} · fecha efectiva {FECHA_EFECTIVA} · {ARCHIVO.name}")

## 2. Extract
Lee la fuente cruda y la guarda **sin cambios** en la capa Bronze.

In [ ]:
def extraer(ruta):
    """Lee la fuente cruda y la deja intacta en la capa Bronze (contiene datos personales: la carpeta data/ no se versiona)."""
    df = extract.leer_csv(ruta)
    medallion.guardar_capa(df, "bronze", f"clientes_snapshot_{SNAPSHOT}")
    print("Columnas:", list(df.columns), "· filas:", len(df))   # sin mostrar datos personales
    return df

## 3. Transform  ✏️ TODO
**Implementa al menos 3 reglas de calidad** (la rúbrica del EP3 y la Actividad 2.2 lo piden). Estas son las que traen los datos de ejemplo:

| Problema en los datos de ejemplo | Dimensión de calidad | Pista |
|---|---|---|
| RUT con dígito verificador incorrecto o texto que no es un RUT (2 filas) | Validez | `q.validar_ruts` |
| Mismo cliente repetido con RUT escrito distinto (con/sin puntos), correo y nombre distintos (2 filas) | Unicidad | `q.anonimizar_serie` normaliza el RUT: luego `duplicated` sobre `rut_anonimizado` |
| Comuna vacía (1 fila) o código que no existe en `dim_geografia` (1 fila) | Completitud / Consistencia | no se rechaza: va al miembro `-1`. Usa `db.mapa_claves` |
| Fecha de nacimiento, nombre y correo no deben llegar al DW | Minimización (Ley 21.719) | `q.tramo_etario` y no incluir esas columnas en la salida |

> Con los datos de ejemplo el snapshot 1 debería dejar fuera 4 filas (2 RUT inválidos y 2 duplicados) como conteo de control.

In [ ]:
# Contrato de salida de transformar(): columnas que espera la carga (no lo borres)
from src.config import exigir_rut_salt
COLUMNAS_SALIDA = ["rut_anonimizado", "tramo_etario", "segmento", "codigo_comuna", "geografia_key", "fecha_primer_registro"]

In [ ]:
def transformar(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Limpia y valida. Retorna (df_limpio, df_rechazos).

    ✏️ TODO (integrante responsable). Pasos sugeridos:
      1. Rechazar RUT inválidos (el `detalle` NO debe incluir el RUT).
      2. `rut_anonimizado = q.anonimizar_serie(df["rut"], exigir_rut_salt())` y deduplicar por esa columna.
      3. Calcular `tramo_etario` (fecha de referencia: `FECHA_EFECTIVA`) y `fecha_primer_registro`.
      4. Resolver `geografia_key` desde la comuna: `db.mapa_claves(engine, "dim_geografia", "codigo_comuna", "geografia_key")`;
         lo que no exista queda en -1 (y `codigo_comuna` en "-1").
      5. Devolver `df_limpio[COLUMNAS_SALIDA]` (una fila por cliente) y los rechazos consolidados.
    """
    raise NotImplementedError("TODO: implementar transformar() de dim_cliente")

## 4. Load
Carga idempotente con el helper compartido y ejecución completa con auditoría (`Ejecucion`).

In [ ]:
ATRIBUTOS_SCD2 = ["codigo_comuna", "segmento"]   # si el cliente se cambia de comuna o de segmento, se versiona
ATRIBUTOS_SCD1 = ["tramo_etario", "fecha_primer_registro", "geografia_key"]

def cargar(df_limpio: pd.DataFrame, df_rechazos: pd.DataFrame) -> dict:
    """Registra los rechazos y carga la dimensión aplicando SCD Tipo 2."""
    assert set(COLUMNAS_SALIDA) <= set(df_limpio.columns), "transformar() no devolvió las columnas esperadas"
    db.registrar_rechazos(engine, PROCESO, df_rechazos)
    return db.aplicar_scd2(engine, "dim_cliente", df_limpio, clave_natural="rut_anonimizado",
                           atributos_scd2=ATRIBUTOS_SCD2, fecha_efectiva=FECHA_EFECTIVA,
                           key_col="cliente_key", atributos_scd1=ATRIBUTOS_SCD1)

In [ ]:
with db.Ejecucion(engine, PROCESO) as run:
    df_crudo = extraer(ARCHIVO)
    run.leidas = len(df_crudo)
    df_limpio, df_rechazos = transformar(df_crudo)
    medallion.guardar_capa(df_limpio, "silver", f"clientes_snapshot_{SNAPSHOT}")
    resumen = cargar(df_limpio, df_rechazos)
    run.cargadas = resumen["nuevos"] + resumen["versiones_nuevas"]
    run.rechazadas = len(df_rechazos)
resumen

## 5. Validación
Chequeos automáticos sobre lo ya cargado. Si algo está mal, el notebook **falla** (no sigue en silencio).

In [ ]:
def validar():
    dim = db.leer_tabla(engine, "dim_cliente")
    problemas = q.chequear_scd2(dim, "rut_anonimizado", "cliente_key")
    assert not problemas, problemas
    assert (dim["cliente_key"] == -1).sum() == 1, "debe existir exactamente un miembro 'No informado' (-1)"
    log.info("Validación de dim_cliente OK (%s filas)", len(dim))

validar()

In [ ]:
# Idempotencia: volver a cargar los MISMOS datos no debe cambiar nada
filas_antes = len(db.leer_tabla(engine, "dim_cliente"))
resumen_2 = cargar(df_limpio, pd.DataFrame(columns=q.COLUMNAS_RECHAZO))
assert len(db.leer_tabla(engine, "dim_cliente")) == filas_antes, "la carga no es idempotente"
print("Idempotencia OK:", resumen_2)

## 6. Evidencia
Estas salidas son las que se capturan para el informe (apartados 6, 7 y 9 de la pauta EP3).

In [ ]:
def conteos_por_capa() -> pd.DataFrame:
    """Filas en cada capa Medallion para este snapshot: trazabilidad fuente → bronze → silver → gold."""
    bronze = medallion.leer_capa("bronze", f"clientes_snapshot_{SNAPSHOT}")
    silver = medallion.leer_capa("silver", f"clientes_snapshot_{SNAPSHOT}")
    gold = db.leer_tabla(engine, "dim_cliente", solo_vigentes=True)
    gold = gold[gold["cliente_key"] != -1]
    return pd.DataFrame({"capa": ["bronze (crudo)", "silver (limpio)", "gold (vigentes en el DW)"],
                          "filas": [len(bronze), len(silver), len(gold)]})

conteos_por_capa()

In [ ]:
# Historia SCD2 de un registro de ejemplo (un cliente que se cambia de comuna en el snapshot 2): una fila por versión
dim = db.leer_tabla(engine, "dim_cliente")
versiones = dim.groupby("rut_anonimizado").size()
ejemplos = versiones[versiones > 1].index[:1]
dim[dim["rut_anonimizado"].isin(ejemplos)].sort_values(["rut_anonimizado", "valido_desde"])

In [ ]:
# Rechazos de este proceso y últimas ejecuciones registradas (auditoría)
rech = db.leer_tabla(engine, "auditoria_rechazos")
display(rech[rech["proceso"] == PROCESO].groupby("motivo_rechazo").size().rename("filas").to_frame())
ejec = db.leer_tabla(engine, "auditoria_ejecuciones")
ejec[ejec["proceso"] == PROCESO].tail(4)[["inicio", "estado", "filas_leidas", "filas_cargadas", "filas_rechazadas"]]

## 7. Decisiones de diseño (completar)
- Es el caso que justificamos con costo-beneficio: ¿por qué SCD2 y no SCD3 para la comuna de residencia? (trazabilidad, Ley 21.719, KPIs históricos).
- `geografia_key` está como SCD1: ¿conviene? ¿Qué pasa si la zona comercial de la comuna cambia después?
- Privacidad: la capa Bronze guarda el RUT crudo. ¿Se anonimiza antes de Bronze o se restringe el acceso a `data/`? Dejar la decisión documentada.